In [13]:
from langchain_core.messages import HumanMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import MessagesState
from langchain_deepseek import ChatDeepSeek
from langchain.chat_models import init_chat_model
from rich import print as rprint
import os
from dotenv import load_dotenv
load_dotenv(override=True)

BAILIAN_API_KEY = os.getenv("BAILIAN_API_KEY")
BAILIAN_API_URL = os.getenv("BAILIAN_API_URL")

#0. 连接LLM模型
model = init_chat_model(
    model = "openai:qwen3.8-max",
    api_key = BAILIAN_API_KEY,
    base_url = BAILIAN_API_URL,
)


#1. 定义状态
class OverAllState(MessagesState):
    username: str
    output: str

#2. 定义节点
def node_a(state: OverAllState) -> OverAllState:
    return {
        "messages": [HumanMessage("你好,我是" + state["username"])],
    }

def llm_node(state: OverAllState) -> OverAllState:
    res = model.invoke(state["messages"])
    return {
        "messages": [res],
        "output": res.content
    }

#3. 构建图
builder = StateGraph(state_schema=OverAllState)
builder.add_node("node_a", node_a)
builder.add_node("llm_node", llm_node)
builder.add_edge(START, "node_a")
builder.add_edge("node_a", "llm_node")
builder.add_edge("llm_node", END)

graph = builder.compile()

#4. 运行图
result = graph.invoke({"username": "老王"})
print(result)


{'messages': [HumanMessage(content='你好,我是老王', additional_kwargs={}, response_metadata={}, id='b2ec6a6b-5065-411c-8482-52ea8861862d'), AIMessage(content='你好，老王！很高兴认识你。有什么我可以帮你的吗？', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 56, 'prompt_tokens': 65, 'total_tokens': 121, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 39, 'rejected_prediction_tokens': None, 'text_tokens': 56}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0, 'text_tokens': 65}}, 'model_provider': 'openai', 'model_name': 'qwen3.8-max', 'system_fingerprint': None, 'id': 'chatcmpl-0d640133-904a-9f00-8c88-eead2013a625', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a18b83-7c12-76e5-a9a3-65111a4308ce-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 65, 'output_tokens': 56, 'total_tokens': 121, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'rea